# Ejercicio 1 — Predice antes de correr

**Consigna:** antes de escribir una sola línea de código, respondé en la celda de texto de abajo:

> Si el dataset `reviews_hoteles.csv` tiene 85% de reseñas positivas y 15% negativas, ¿qué accuracy esperás que tenga un modelo que **siempre** responde "positivo", sin haber aprendido nada?

Escribí tu predicción con un número aproximado. Después vas a verificarla en el Ejercicio 4.

### Mi predicción:

*(completá acá antes de seguir)*


# Ejercicio 2 — Pipeline completo con un dataset nuevo

**Consigna:** `reviews_hoteles.csv` tiene reseñas de hoteles (otro rubro, mismo problema: clasificar sentimiento). Armá el pipeline **completo** desde cero.

1. Cargar el dataset.
2. Split train/test.
3. Vectorizar con TF-IDF (`fit_transform` en train, `transform` en test — ¿te acordás por qué en ese orden?).
4. Entrenar un `LogisticRegression`.
5. Calcular accuracy.
6. Probar el modelo con 2 reseñas de hotel que inventes tu.

In [1]:
import pandas as pd

df_hoteles = pd.read_csv("/workspaces/data-analysis-course/modulo-4-visualizacion/clase-16/practica/reviews_hoteles.csv")

reseñas = df_hoteles['texto']
etiquetas = df_hoteles['sentimiento']

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    reseñas,
    etiquetas,
    test_size=0.2,
    random_state=42,
    stratify=etiquetas
)


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [4]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression()

modelo.fit(X_train_vec, y_train)

predicciones = modelo.predict(X_test_vec)

print(predicciones[:5])

print(modelo.score(X_test_vec, y_test))

['positivo' 'negativo' 'negativo' 'negativo' 'positivo']
0.375


In [5]:

from sklearn.metrics import confusion_matrix, classification_report

predicciones = modelo.predict(X_test_vec)

print(confusion_matrix(y_test, predicciones))
print(classification_report(y_test, predicciones))


[[1 3]
 [2 2]]
              precision    recall  f1-score   support

    negativo       0.33      0.25      0.29         4
    positivo       0.40      0.50      0.44         4

    accuracy                           0.38         8
   macro avg       0.37      0.38      0.37         8
weighted avg       0.37      0.38      0.37         8



In [6]:
# TODO: probá el modelo con 2 reseñas de hotel inventadas por vos
# Recordá: hay que vectorizarlas con vectorizer.transform() (no fit_transform)

reseñas_nuevas = [
    "Muy cómodo, limpio y luminoso, lo recomiendo",
    "Pésima habitación, no se parece a lo ofrecido en la imagen",
    "Excelente servicio y muy buena ubicación",
    "La atención fue maravillosa y todo estaba impecable",
    "Muy ruidoso y la atención fue fatal",
    "El baño estaba sucio y no había agua caliente"
]
#1 = positiva, 0 = negativa
etiquetas = [1,0,1,1,0,0]

X_train, X_test, y_train, y_test = train_test_split(
    reseñas_nuevas,
    etiquetas,
    test_size=0.33,
    random_state=42,
    stratify=etiquetas
)

print("Entrenamiento:", X_train)
print("Test:", X_test)


Entrenamiento: ['Muy ruidoso y la atención fue fatal', 'La atención fue maravillosa y todo estaba impecable', 'Pésima habitación, no se parece a lo ofrecido en la imagen', 'Excelente servicio y muy buena ubicación']
Test: ['Muy cómodo, limpio y luminoso, lo recomiendo', 'El baño estaba sucio y no había agua caliente']


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Vectorización (Recordando usar fit_transform en train y transform en test)
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 2. Inicializar el modelo de Regresión Logística
modelo = LogisticRegression(random_state=42)

# 3. Entrenar el modelo con los datos de entrenamiento
modelo.fit(X_train_vec, y_train)

# 4. Probar / Predecir con los datos de test
predicciones = modelo.predict(X_test_vec)

print("Predicciones del modelo:", predicciones)
print("Etiquetas reales (y_test):", list(y_test))

Predicciones del modelo: [0 1]
Etiquetas reales (y_test): [1, 0]


In [9]:
from sklearn.metrics import confusion_matrix, classification_report

predicciones = modelo.predict(X_test_vec)

print(confusion_matrix(y_test, predicciones))
print(classification_report(y_test, predicciones))

[[0 1]
 [1 0]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       1.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



## Ejercicio 3 — Duelo: tu modelo (Ejercicio 2) vs. pysentimiento

**Consigna:** vas a usar el modelo que corregiste en el Ejercicio 2 —entrenado sobre `reviews_base.csv`, sin data leakage— y compararlo contra pysentimiento en 10 reseñas con sarcasmo, negaciones y opiniones mixtas.

Usamos justamente el modelo del Ejercicio 2 y no el de hoteles del Ejercicio 3 porque `reviews_casos_limite.csv` es del mismo dominio (e-commerce) que `reviews_base.csv`. Comparar contra un modelo entrenado en otro rubro no sería una prueba justa.

1. Reentrená el pipeline corregido del Ejercicio 2 (podés copiar tu solución, o completar el esqueleto de acá abajo).
2. Instalá pysentimiento (`pip install pysentimiento`) y corré las mismas 10 reseñas.
3. Armá una tabla comparando: texto | predicción de tu modelo | predicción de pysentimiento | tu propio juicio como humano.
4. Marcá en qué casos discrepan los dos modelos.
5. Reflexión final (2-3 líneas): en los casos donde discreparon, ¿con cuál coincidís vos como humano? Pensá en el tamaño de cada dataset de entrenamiento — `reviews_base.csv` tiene 120 filas, pysentimiento fue entrenado con muchísimas más. ¿Cómo se relaciona eso con que uno de los dos acierte más seguido en casos ambiguos?



In [10]:
pip install pysentimiento


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
df = pd.read_csv("/workspaces/data-analysis-course/modulo-4-visualizacion/clase-16/practica/reviews_base.csv")
df

reseñas = df['texto']
etiquetas = df['sentimiento']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    reseñas,
    etiquetas,
    test_size=0.2,
    random_state=42,
    stratify=etiquetas
)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression()

modelo.fit(X_train_vec, y_train)

predicciones = modelo.predict(X_test_vec)

print(predicciones[:5])

print(modelo.score(X_test_vec, y_test))


from sklearn.metrics import confusion_matrix, classification_report

predicciones = modelo.predict(X_test_vec)

print(confusion_matrix(y_test, predicciones))
print(classification_report(y_test, predicciones))

['positivo' 'positivo' 'positivo' 'positivo' 'negativo']
1.0
[[12  0]
 [ 0 12]]
              precision    recall  f1-score   support

    negativo       1.00      1.00      1.00        12
    positivo       1.00      1.00      1.00        12

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



In [ ]:
# TODO: predicciones con tu propio modelo (recordá vectorizer.transform, no fit_transform)


In [15]:
# pip install pysentimiento
from pysentimiento import create_analyzer

analyzer = create_analyzer(task="sentiment", lang="es")

# TODO: corré analyzer.predict() sobre cada texto de df_limite y guardá el resultado


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 120.14it/s]


In [18]:
df_base = pd.read_csv("reviews_base.csv")
print(df_base)

                                            texto sentimiento
0                         Se rompió al primer uso    negativo
1                  Buena atención y envío cuidado    positivo
2                    Gran relación precio-calidad    positivo
3                  Terrible experiencia de compra    negativo
4                      Pésima atención al cliente    negativo
..                                            ...         ...
115              Todo perfecto de principio a fin    positivo
116                         Una pérdida de dinero    negativo
117                  Gran relación precio-calidad    positivo
118    Tardaron demasiado en responder mi reclamo    negativo
119  Excelente producto, lo recomiendo totalmente    positivo

[120 rows x 2 columns]


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# 1. Cargar datasets
df_base = pd.read_csv("reviews_base.csv")
df_limite = pd.read_csv("reviews_casos_limite.csv")

# 2. Reentrenar el pipeline corregido del Ejercicio 2
# (Ajustá los componentes según lo que hayas elegido en el Ejercicio 2)
pipeline_ej2 = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    LogisticRegression()
)

pipeline_ej2.fit(df_base["texto"], df_base["sentimiento"])

# Predicción de tu modelo (Ejercicio 2)
df_limite["pred_modelo_ej2"] = pipeline_ej2.predict(df_limite["texto"])

# Predicción de pysentimiento
# analyzer.predict() devuelve un objeto SentimentOutput; obtenemos la etiqueta con .output
df_limite["pred_pysentimiento"] = df_limite["texto"].apply(
    lambda x: analyzer.predict(x).output
)

In [ ]:
# Mapeo opcional para homogeneizar etiquetas si tu modelo usaba "positivo"/"negativo"
mapa_etiquetas = {"POS": "positivo", "NEG": "negativo", "NEU": "neutro"}
df_limite["pred_pysentimiento_norm"] = df_limite["pred_pysentimiento"].map(
    lambda x: mapa_etiquetas.get(x, x)
)

# Marcamos en qué casos discrepan
df_limite["discrepan"] = (
    df_limite["pred_modelo_ej2"] != df_limite["pred_pysentimiento_norm"]
)

# Tu juicio humano (completar manualmente o inspeccionar fila a fila)
# df_limite['juicio_humano'] = [...]

# Visualización limpia de la tabla
columnas_tabla = [
    "texto",
    "pred_modelo_ej2",
    "pred_pysentimiento_norm",
    "discrepan"
]
print(df_limite[columnas_tabla])

                                               texto pred_modelo_ej2  \
0  Que buenísimo, llegó roto justo el día que lo ...        positivo   
1          No está nada mal para el precio que pagué        negativo   
2  No es que sea malo, pero tampoco lo compraría ...        negativo   
3      Increíble... tres semanas de espera para esto        negativo   
4  El diseño me encanta pero la calidad deja bast...        negativo   
5          No me arrepiento para nada de esta compra        negativo   
6           Justo lo que necesitaba, ni más ni menos        positivo   
7        Funciona, pero no esperen ninguna maravilla        positivo   
8         Nada que objetar, todo salió como esperaba        positivo   
9  Un producto que no defrauda ni sorprende, cump...        negativo   

  pred_pysentimiento_norm  discrepan  
0                negativo       True  
1                positivo       True  
2                  neutro       True  
3                negativo      False  
4           

In [ ]:
# TODO: armá la tabla comparativa final
# columnas sugeridas: texto, prediccion_propia, prediccion_pysentimiento, mi_juicio
#texto | predicción propia | predicción pysentimiento | mi juicio

#texto           |            predicción propia      |           predicción pysentimiento        |        mi juicio
#"que buenísimo, llegó roto justo el día que lo necesitaba | positivo | negativo | el modelo no detecta sarcasmos
#"no está nada mal para el precio que pagué" | negativo | positivo | el modelo interpreta palabras como "mal" como reseñas negativas
#"justo lo que necesitaba, ni más ni menos | positivo | positivo | el modelo pudo interpretar correctamente un mensaje que no tiene grandes ambiguedades
#"no me arrepiento para nada de esta compra" | negativo | positivo | la reseña puede cambiar de significado al pasar por ciertos "filtros" del modelo, por ejemplo si borra la palabra "no"

### Tu reflexión:
Para una mayor precisión del modelo, necesitaríamos entrenarlo con miles y miles de oraciones y diferentes casos y combinaciones de palabras. Como vimos, es posible lograr que incluso entienda el uso de sarcasmo y ambiguedades, pero eso se ha logrado con el análisis de millones de textos. 
